# 近傍語の入れ替わりで意味の変化を測る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_lexical_semantics/semantic_change_neighbors.ipynb)

NLP クックブック 18-10 ／ 語の意味と意味の変化

単語ベクトルの位置合わせをせずに、近い語の顔ぶれがどれだけ入れ替わったかで意味の変化を測りたい。

In [1]:
!pip install -q datasets fugashi unidic-lite gensim

In [2]:
import random, re
import numpy as np
from datasets import load_dataset
from fugashi import Tagger
from gensim.models import Word2Vec

ds = load_dataset("globis-university/aozorabunko-clean", split="train")
tagger = Tagger()
random.seed(0)

def era(meta):
    year = (meta.get("生年月日") or "")[:4]
    if meta["文字遣い種別"] != "新字新仮名" or not year.isdigit():
        return None
    return "旧" if 1850 <= int(year) < 1880 else "新" if 1895 <= int(year) < 1930 else None

texts, common_nouns = {"旧": [], "新": []}, set()
for ex in ds:
    if era(ex["meta"]):
        texts[era(ex["meta"])].append(ex["text"])

def tokenize(works, max_chars=20_000_000):
    sentences, n = [], 0
    for text in random.sample(works, len(works)):
        for s in re.split(r"(?<=[。！？])|\n", text[:30000]):
            if len(s.strip()) >= 5:
                words = tagger(s.strip())
                common_nouns.update(w.surface for w in words if w.feature.pos2 == "普通名詞")
                sentences.append([w.surface for w in words]); n += len(s)
        if n > max_chars:
            break
    return sentences

models = {e: Word2Vec(tokenize(t), vector_size=100, window=5, min_count=30, sg=1, epochs=3, workers=4, seed=0).wv for e, t in texts.items()}
vocab = [w for w in models["旧"].index_to_key if w in models["新"].key_to_index
         and models["旧"].get_vecattr(w, "count") >= 100 and models["新"].get_vecattr(w, "count") >= 100]

In [3]:
K = 25
shared = set(vocab)

def neighbors(wv, word, k=K):
    return [w for w, _ in wv.most_similar(word, topn=k * 4) if w in shared][:k]      # 両方の時代にある語だけを近傍とする

def jaccard_change(word):
    """近傍語の集合がどれだけ入れ替わったか (1 − Jaccard 係数)"""
    a, b = set(neighbors(models["旧"], word)), set(neighbors(models["新"], word))
    return 1 - len(a & b) / len(a | b)

def second_order_change(word):
    """両時代の近傍語を合わせた集合に対する類似度の並びを比べる。同じ語の集合を物差しにするので、位置合わせが要らない"""
    ref = sorted(set(neighbors(models["旧"], word)) | set(neighbors(models["新"], word)))
    a = np.array([models["旧"].similarity(word, r) for r in ref])
    b = np.array([models["新"].similarity(word, r) for r in ref])
    return 1 - float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))

for w in ["原子", "同志", "活動", "戦争", "言葉", "必要"]:
    print(f"{w}: 近傍の入れ替わり {jaccard_change(w):.2f} / 二次の類似度による変化 {second_order_change(w):.3f}")

原子: 近傍の入れ替わり 0.90 / 二次の類似度による変化 0.031
同志: 近傍の入れ替わり 0.91 / 二次の類似度による変化 0.020
活動: 近傍の入れ替わり 0.96 / 二次の類似度による変化 0.008
戦争: 近傍の入れ替わり 0.86 / 二次の類似度による変化 0.019
言葉: 近傍の入れ替わり 0.78 / 二次の類似度による変化 0.016
必要: 近傍の入れ替わり 0.68 / 二次の類似度による変化 0.008


In [4]:
# 位置合わせによる方法 (直交プロクラステス) と、どれくらい同じ結果になるか
from scipy.stats import spearmanr

A = np.stack([models["旧"][w] for w in vocab]); A /= np.linalg.norm(A, axis=1, keepdims=True)
B = np.stack([models["新"][w] for w in vocab]); B /= np.linalg.norm(B, axis=1, keepdims=True)
U, _, Vt = np.linalg.svd(A.T @ B)
procrustes = dict(zip(vocab, 1 - ((A @ U @ Vt) * B).sum(axis=1)))

targets = [w for w in vocab if w in common_nouns and re.fullmatch(r"[一-龥ァ-ヶー]{2,}", w)]
sample = random.sample(targets, 500)
local = {w: second_order_change(w) for w in sample}
print("順位相関:", round(spearmanr([procrustes[w] for w in sample], [local[w] for w in sample]).statistic, 3))
print("近傍で見て変化が大きい:", sorted(sample, key=local.get, reverse=True)[:20])
print("位置合わせで見て変化が大きい:", sorted(sample, key=procrustes.get, reverse=True)[:20])

順位相関: 0.569
近傍で見て変化が大きい: ['名人', 'レコード', '浮浪', '行方', '天下', '家人', '一刻', '一口', '知恵', '一目', '先方', '太夫', '映画', '今宵', '最後', '太子', '主人', '攻撃', '信者', '露出']
位置合わせで見て変化が大きい: ['名人', '浮浪', 'レコード', '家人', '公爵', '太夫', '玉子', '知恵', '信者', '十字', '中途', '戦場', '小娘', '王子', '主人', '八方', '和尚', '百姓', '悪魔', '天下']


- 近傍語の比較は、ベクトル空間どうしを重ねる必要がない。別々のモデル、別々の次元数、別々の手法で作ったベクトルでも比べられる。
- 位置合わせによる方法は、文脈全体の変化（連想の変化など）を広く拾う。近傍語による方法は、入れ替え可能な語の変化、つまり語義そのものの変化に敏感だとされている。
- 近傍の数 K によって結果が変わる。小さいと偶然の入れ替わりに左右され、大きいと変化が薄まる。
- 2 つの方法の順位はある程度一致するが、同じではない。複数の指標で上位に来る語から調べるのが堅実。